# LeetCode 3878 — Count Good Subarrays bằng Python

- Link: https://leetcode.com/problems/count-good-subarrays/
- Độ khó: **Hard**
- Chủ đề: Phép OR bit, quan hệ "tập con bit", ngăn xếp đơn điệu (monotonic stack), đếm theo đóng góp (contribution), tập OR của các đoạn kết thúc tại một vị trí

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countGoodSubarrays(self, nums: list[int]) -> int:
```

## 1. Mục tiêu học

1. Hiểu điều kiện "OR của đoạn **bằng một phần tử trong đoạn**" ⇔ có phần tử `x` trong đoạn mà mọi phần tử khác là **tập con bit** của `x` (`y | x == x`).
2. Đếm theo **đóng góp**: mỗi đoạn tốt được gán cho đúng một "nhân chứng" `i`, rồi đếm số đoạn nhận `i` làm nhân chứng = `(i - L) · (R - i)`.
3. Xử lý **phần tử trùng giá trị** để không đếm lặp (chọn nhân chứng là lần xuất hiện **trái nhất**).
4. Tìm biên `L`, `R` bằng **ngăn xếp đơn điệu** theo quan hệ tập con — `O(n)`.
5. Một hướng khác: mỗi vị trí phải có tối đa ~31 giá trị OR khác nhau của các đoạn kết thúc tại đó.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `nums`. Một đoạn con (liên tiếp, khác rỗng) là **tốt** nếu OR bit của mọi phần tử trong đoạn **bằng ít nhất một phần tử** của đoạn. Đếm số đoạn tốt.

**Ví dụ 1:** `nums = [4,2,3]` → `4`. Tốt: `[4]`, `[2]`, `[3]`, `[2,3]` (OR = 3). Không tốt: `[4,2]` (OR 6), `[4,2,3]` (OR 7).

**Ví dụ 2:** `nums = [1,3,1]` → `6` (mọi đoạn đều tốt: có 3 thì OR = 3; chỉ có 1 thì OR = 1).

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `0 ≤ nums[i] ≤ 10^9`.

In [1]:
import random, time

examples = [([4, 2, 3], 4), ([1, 3, 1], 6)]
for nums, exp in examples:
    print(nums, "-> kỳ vọng", exp)

[4, 2, 3] -> kỳ vọng 4
[1, 3, 1] -> kỳ vọng 6


## 3. Kiến thức nền

### 3.1. Phép OR và quan hệ tập con bit

`y` là **tập con bit** của `x` (mọi bit 1 của `y` đều là bit 1 của `x`) ⇔ `y | x == x` ⇔ `y & ~x == 0`. Nếu `y ⊆ x` thì `y ≤ x` (về giá trị).

In [2]:
def subset(y, x):
    return y | x == x

print(subset(2, 3), subset(4, 3), subset(0, 5), subset(5, 5))
print(bin(4), bin(2), bin(3), "| 4|2 =", 4 | 2, "| 2|3 =", 2 | 3)

True False True True
0b100 0b10 0b11 | 4|2 = 6 | 2|3 = 3


### 3.2. OR của đoạn bằng một phần tử ⇔ phần tử đó "chứa" mọi phần tử khác

Nếu OR của đoạn là `V` và `V = nums[i]` với `i` trong đoạn, thì mọi phần tử `⊆ V`. Ngược lại nếu có `x` trong đoạn chứa mọi phần tử khác thì OR = `x`. Hệ quả: phần tử đó cũng là **max** của đoạn ⇒ "đoạn tốt" ⇔ "OR = max của đoạn".

In [3]:
from functools import reduce
for seg in ([4, 2], [2, 3], [1, 3, 1], [5, 1, 4]):
    o = reduce(lambda a, b: a | b, seg)
    print(seg, "OR =", o, "| max =", max(seg), "| tốt?", o in seg, "| OR == max?", o == max(seg))

[4, 2] OR = 6 | max = 4 | tốt? False | OR == max? False
[2, 3] OR = 3 | max = 3 | tốt? True | OR == max? True
[1, 3, 1] OR = 3 | max = 3 | tốt? True | OR == max? True
[5, 1, 4] OR = 5 | max = 5 | tốt? True | OR == max? True


### 3.3. Ngăn xếp đơn điệu — "phần tử gần nhất bên trái không thỏa"

Mẫu chuẩn: duyệt trái → phải, pop các chỉ số trên đỉnh **thỏa** điều kiện so với phần tử hiện tại; đỉnh còn lại (nếu có) là phần tử gần nhất **không thỏa**. Ví dụ quen thuộc: "phần tử lớn hơn gần nhất bên trái".

In [4]:
def prev_greater(a):
    st, res = [], []
    for i, x in enumerate(a):
        while st and a[st[-1]] <= x:
            st.pop()
        res.append(st[-1] if st else -1)
        st.append(i)
    return res

print(prev_greater([3, 1, 4, 1, 5, 9, 2, 6]))

[-1, 0, -1, 2, -1, -1, 5, 5]


### 3.4. Đếm đoạn theo đóng góp

Nếu `i` là nhân chứng của mọi đoạn `[l, r]` với `L < l ≤ i ≤ r < R`, số đoạn đó là `(i - L) · (R - i)`. Tổng đóng góp mọi `i` = đáp án, **miễn là** mỗi đoạn tốt được gán cho **đúng một** `i`.

In [5]:
L, i, R = 1, 4, 7
print("số đoạn chứa i trong (L, R):", (i - L) * (R - i),
      "| liệt kê:", len([(l, r) for l in range(L + 1, i + 1) for r in range(i, R)]))

số đoạn chứa i trong (L, R): 9 | liệt kê: 9


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Gọi `V` là OR của một đoạn tốt; `V` xuất hiện trong đoạn (có thể nhiều lần). Ta gán đoạn cho **lần xuất hiện trái nhất** của `V` trong đoạn — gọi là `i`. Khi đó:

- mọi phần tử trong đoạn là tập con của `nums[i]`;
- các phần tử **bên trái** `i` trong đoạn là tập con **thực sự** (khác `nums[i]`), vì `i` là lần xuất hiện trái nhất;
- các phần tử **bên phải** `i` là tập con (được phép bằng).

Tính tay ví dụ 1, `[4, 2, 3]`:

| i | nums[i] | mở trái tới | mở phải tới | đóng góp |
|---|---|---|---|---|
| 0 | 4 (100) | biên trái | 2 ⊄ 4 ⇒ dừng | 1 · 1 = 1 |
| 1 | 2 (010) | 4 ⊄ 2 ⇒ dừng | 3 ⊄ 2 ⇒ dừng | 1 |
| 2 | 3 (011) | 2 ⊂ 3 ✔, 4 ⊄ 3 ⇒ dừng | biên | 2 · 1 = 2 |

Tổng `4` ✔.

## 5. Cách 1 — Brute Force: duyệt mọi đoạn

Cố định đầu trái `l`, mở rộng `r`, cập nhật OR và tập các giá trị trong đoạn. `O(n²)` thời gian.

In [6]:
def brute(nums):
    n = len(nums)
    cnt = 0
    for l in range(n):
        o = 0
        seen = set()
        for r in range(l, n):
            o |= nums[r]
            seen.add(nums[r])
            if o in seen:
                cnt += 1
    return cnt

for nums, exp in examples:
    print(brute(nums), "kỳ vọng", exp)

4 kỳ vọng 4
6 kỳ vọng 6


**Nút thắt:** `n²/2 = 5·10^9` đoạn với `n = 10^5`. Cần đếm "theo nhân chứng" thay vì theo đoạn.

## 6. Ý tưởng thuật toán / bước đột phá

Với mỗi `i`, định nghĩa:

- `L[i]` = chỉ số **gần nhất bên trái** mà `nums[L]` **không** là tập con thực sự của `nums[i]` (tức `nums[L] ⊄ nums[i]` **hoặc** `nums[L] == nums[i]`), hoặc `-1`;
- `R[i]` = chỉ số **gần nhất bên phải** mà `nums[R] ⊄ nums[i]`, hoặc `n`.

Số đoạn nhận `i` làm nhân chứng (trái nhất) là `(i - L[i]) · (R[i] - i)`. Đáp án `= Σ_i (i - L[i]) · (R[i] - i)`.

**Đột phá — ngăn xếp đơn điệu theo quan hệ tập con:** duyệt trái → phải, pop các `j` trên đỉnh với `nums[j] ⊊ nums[i]`. Đúng vì quan hệ tập con **bắc cầu**: nếu `j` bị pop bởi `k` (`nums[j] ⊆ nums[k]`), thì với `i` sau này:

- nếu `nums[k] ⊆ nums[i]` thì `nums[j] ⊆ nums[i]` — `j` cũng không phải biên;
- nếu `nums[k] ⊄ nums[i]` thì `k` là biên gần hơn `j`.

Vậy `j` không bao giờ cần nữa. Phía phải làm tương tự (pop khi `nums[j] ⊆ nums[i]`, kể cả bằng).

## 7. Các cách tiếp theo

- **Cách 2:** với mỗi `i`, mở rộng biên trái/phải bằng vòng lặp — `O(n²)` tệ nhất, nhanh khi dữ liệu "ngẫu nhiên".
- **Cách 3:** ngăn xếp đơn điệu cho `L`, `R` — `O(n)` (lời giải nộp chính).
- **Cách 4:** với mỗi đầu phải `r`, nhóm các đầu trái theo giá trị OR (≤ 31 nhóm) và dùng vị trí xuất hiện gần nhất của giá trị — `O(n log V)`.

### 7.1. Cách 2 — Mở rộng biên trực tiếp

In [7]:
def expand(nums):
    n = len(nums)
    total = 0
    for i, x in enumerate(nums):
        L = i - 1
        while L >= 0 and nums[L] | x == x and nums[L] != x:
            L -= 1
        R = i + 1
        while R < n and nums[R] | x == x:
            R += 1
        total += (i - L) * (R - i)
    return total

for nums, exp in examples:
    print(expand(nums), "kỳ vọng", exp)

4 kỳ vọng 4
6 kỳ vọng 6


**Độ phức tạp:** `O(n²)` tệ nhất (ví dụ mảng toàn số `0` và một số lớn…), `O(n)` bộ nhớ.

### 7.2. Cách 3 — Ngăn xếp đơn điệu (lời giải chính)

**Giải thích từng đoạn:**

1. Trái → phải: pop đỉnh `j` khi `nums[j] | x == x` **và** `nums[j] != x` (tập con thực sự). `L[i]` = đỉnh còn lại hoặc `-1`. Đẩy `i`.
2. Phải → trái: pop đỉnh `j` khi `nums[j] | x == x` (kể cả bằng). `R[i]` = đỉnh còn lại hoặc `n`. Đẩy `i`.
3. Cộng `(i - L[i]) · (R[i] - i)`.

(Điều kiện `nums[j] < x` mà doocs dùng tương đương `nums[j] != x` khi đã có `nums[j] ⊆ x`.)

In [8]:
def mono_stack(nums):
    n = len(nums)
    L = [-1] * n
    st = []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] | x == x and nums[st[-1]] != x:
            st.pop()
        L[i] = st[-1] if st else -1
        st.append(i)
    R = [n] * n
    st = []
    for i in range(n - 1, -1, -1):
        x = nums[i]
        while st and nums[st[-1]] | x == x:
            st.pop()
        R[i] = st[-1] if st else n
        st.append(i)
    return sum((i - L[i]) * (R[i] - i) for i in range(n))

for nums, exp in examples:
    print(mono_stack(nums), "kỳ vọng", exp)

4 kỳ vọng 4
6 kỳ vọng 6


**Độ phức tạp Cách 3:** mỗi chỉ số vào/ra ngăn xếp một lần mỗi chiều ⇒ `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.3. Cách 4 — Tập OR theo đầu phải

**Ý tưởng:** với đầu phải `r`, các giá trị `OR(nums[l..r])` khi `l` giảm dần chỉ **thêm bit** ⇒ có tối đa ~31 giá trị khác nhau, mỗi giá trị ứng với một **khoảng** `l`. Giữ danh sách `groups = [(V, l_min, l_max)]`.

Đoạn `[l, r]` với OR = `V` là tốt ⇔ `V` xuất hiện trong `nums[l..r]` ⇔ `last[V] ≥ l`, với `last[V]` = vị trí gần nhất `≤ r` có giá trị `V`. Vậy trong nhóm `(V, a, b)` số `l` hợp lệ là `max(0, min(b, last[V]) - a + 1)`.

In [9]:
def or_groups(nums):
    last = {}
    groups = []                 # danh sách (V, a, b): mọi l trong [a, b] có OR(nums[l..r]) = V
    total = 0
    for r, x in enumerate(nums):
        last[x] = r
        new = []
        for V, a, b in groups:
            nv = V | x
            if new and new[-1][0] == nv:
                new[-1] = (nv, new[-1][1], b)          # gộp: giữ đầu trái của nhóm trước, mở rộng tới b
            else:
                new.append((nv, a, b))
        if new and new[-1][0] == x:
            new[-1] = (x, new[-1][1], r)
        else:
            new.append((x, r, r))
        groups = new
        for V, a, b in groups:
            p = last.get(V, -1)
            if p >= a:
                total += min(b, p) - a + 1
    return total

for nums, exp in examples:
    print(or_groups(nums), "kỳ vọng", exp)

4 kỳ vọng 4
6 kỳ vọng 6


Lưu ý thứ tự: `groups` được giữ theo `l` **tăng dần** (OR giảm dần theo nghĩa tập con khi `l` tăng). Khi thêm `x`, các nhóm cũ OR thêm `x`; nhóm có cùng giá trị kề nhau được gộp.

In [10]:
class SolutionGroups:
    def countGoodSubarrays(self, nums: list[int]) -> int:
        return or_groups(nums)

print([SolutionGroups().countGoodSubarrays(a) for a, _ in examples])

[4, 6]


**Độ phức tạp Cách 4:** mỗi `r` có `O(log V)` nhóm (`V ≤ 10^9` ⇒ ≤ 31) ⇒ `O(n log V)`, bộ nhớ `O(n)` cho `last`.

## 8. TRACE từng bước

Trace Cách 3 trên `nums = [1, 3, 1, 2, 3]` (tự dựng).

Trái → phải (pop khi tập con **thực sự**):

| i | x | ngăn xếp trước | pop | L[i] | ngăn xếp sau |
|---|---|---|---|---|---|
| 0 | 1 | [] | – | −1 | [0] |
| 1 | 3 | [0] | 0 (1 ⊊ 3) | −1 | [1] |
| 2 | 1 | [1] | – (3 ⊄ 1) | 1 | [1, 2] |
| 3 | 2 | [1, 2] | – (1 ⊄ 2) | 2 | [1, 2, 3] |
| 4 | 3 | [1, 2, 3] | 3, 2; dừng ở 1 (3 == 3) | 1 | [1, 4] |

Hàm trace in cả hai chiều và đóng góp.

In [11]:
def trace(nums):
    n = len(nums); L = [-1] * n; st = []
    for i, x in enumerate(nums):
        popped = []
        while st and nums[st[-1]] | x == x and nums[st[-1]] != x:
            popped.append(st.pop())
        L[i] = st[-1] if st else -1; st.append(i)
        print(f"trái->phải i={i} x={x}: pop {popped} -> L={L[i]}, stack={st}")
    R = [n] * n; st = []
    for i in range(n - 1, -1, -1):
        x = nums[i]; popped = []
        while st and nums[st[-1]] | x == x:
            popped.append(st.pop())
        R[i] = st[-1] if st else n; st.append(i)
        print(f"phải->trái i={i} x={x}: pop {popped} -> R={R[i]}, stack={st}")
    contrib = [(i - L[i]) * (R[i] - i) for i in range(n)]
    print("L =", L, "| R =", R, "| đóng góp =", contrib, "| tổng =", sum(contrib), "| vét cạn =", brute(nums))

trace([1, 3, 1, 2, 3])

trái->phải i=0 x=1: pop [] -> L=-1, stack=[0]
trái->phải i=1 x=3: pop [0] -> L=-1, stack=[1]
trái->phải i=2 x=1: pop [] -> L=1, stack=[1, 2]
trái->phải i=3 x=2: pop [] -> L=2, stack=[1, 2, 3]
trái->phải i=4 x=3: pop [3, 2] -> L=1, stack=[1, 4]
phải->trái i=4 x=3: pop [] -> R=5, stack=[4]
phải->trái i=3 x=2: pop [] -> R=4, stack=[4, 3]
phải->trái i=2 x=1: pop [] -> R=3, stack=[4, 3, 2]
phải->trái i=1 x=3: pop [2, 3, 4] -> R=5, stack=[1]
phải->trái i=0 x=1: pop [] -> R=1, stack=[1, 0]
L = [-1, -1, 1, 2, 1] | R = [1, 5, 3, 4, 5] | đóng góp = [1, 8, 1, 1, 3] | tổng = 14 | vét cạn = 14


## 9. Code nộp LeetCode

In [12]:
class Solution:
    def countGoodSubarrays(self, nums: list[int]) -> int:
        n = len(nums)
        L = [-1] * n
        st = []
        for i, x in enumerate(nums):
            while st:
                y = nums[st[-1]]
                if y | x == x and y != x:     # y là tập con THỰC SỰ của x
                    st.pop()
                else:
                    break
            L[i] = st[-1] if st else -1
            st.append(i)
        R = [n] * n
        st = []
        for i in range(n - 1, -1, -1):
            x = nums[i]
            while st and nums[st[-1]] | x == x:   # tập con (kể cả bằng)
                st.pop()
            R[i] = st[-1] if st else n
            st.append(i)
        return sum((i - L[i]) * (R[i] - i) for i in range(n))

sol = Solution()
for nums, exp in examples:
    got = sol.countGoodSubarrays(nums)
    print(nums, "->", got, "✔" if got == exp else "✘")

[4, 2, 3] -> 4 ✔
[1, 3, 1] -> 6 ✔


## 10. Vì sao thuật toán đúng

1. **Gán duy nhất:** mỗi đoạn tốt có OR `V` xuất hiện trong đoạn; nhân chứng là lần xuất hiện trái nhất `i` của `V`. Ngược lại, mỗi cặp `(đoạn, i)` mà `L[i] < l ≤ i ≤ r < R[i]` cho một đoạn có mọi phần tử `⊆ nums[i]`, các phần tử trái `i` khác `nums[i]` ⇒ OR = `nums[i]` và `i` là lần xuất hiện trái nhất. Đây là song ánh ⇒ tổng `(i - L)(R - i)` đếm mỗi đoạn tốt đúng một lần.
2. **Ngăn xếp đúng:** bất biến — sau khi xử lý `i`, ngăn xếp chứa các chỉ số mà không có phần tử nào phía sau "chứa" nó theo quan hệ pop. Quan hệ tập con bắc cầu (mục 6) bảo đảm phần tử bị pop không bao giờ là biên gần nhất cho các `i` sau. Phía trái pop "tập con thực sự", phía phải pop "tập con" — hai quan hệ đều bắc cầu (tập con thực sự ∘ tập con thực sự là tập con thực sự).
3. **Cách 4:** với `r` cố định, `OR(l..r)` là hàm không tăng (theo tập con) khi `l` tăng, nên các `l` cùng OR tạo khoảng liên tiếp; điều kiện "`V` xuất hiện trong `[l, r]`" ⇔ `last[V] ≥ l`.

## 11. Phân tích Time / Space Complexity

Đặt `n = len(nums)`, `B = 30` (số bit).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn đoạn | `O(n²)` | `O(n)` |
| 2. Mở rộng biên | `O(n²)` tệ nhất | `O(n)` |
| 3. Ngăn xếp đơn điệu | `O(n)` | `O(n)` |
| 4. Nhóm OR theo đầu phải | `O(n · B)` | `O(n)` |

## 12. Bộ test

In [13]:
def run_all(nums, with_brute=True):
    res = [expand(nums), mono_stack(nums), or_groups(nums), Solution().countGoodSubarrays(nums)]
    if with_brute:
        res.append(brute(nums))
    return res

for nums, exp in examples:
    assert all(v == exp for v in run_all(nums))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [14]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng vét cạn)
cases = [
    ([0], 1),
    ([0, 0, 0], 6),            # mọi đoạn: OR = 0 có mặt
    ([7, 7], 3),               # giá trị trùng: không đếm lặp
    ([1, 2, 4, 8], 4),         # các bit rời nhau: chỉ đoạn độ dài 1
    ([3, 1, 2], 5),            # [1,2] có OR = 3 nhưng 3 không nằm trong đoạn
]
for nums, exp in cases:
    r = run_all(nums)
    print(nums, "->", r[0], "| vét cạn:", r[-1])
    assert len(set(r)) == 1 and r[0] == exp

[0] -> 1 | vét cạn: 1
[0, 0, 0] -> 6 | vét cạn: 6
[7, 7] -> 3 | vét cạn: 3
[1, 2, 4, 8] -> 4 | vét cạn: 4
[3, 1, 2] -> 5 | vét cạn: 5


Ca `[3, 1, 2]`: các đoạn `[3]`, `[1]`, `[2]`, `[3,1]`, `[3,1,2]` tốt; `[1,2]` có OR `3` nhưng `3` không nằm trong đoạn ⇒ không tốt. Đáp án `5` — một cái bẫy dễ đoán nhầm thành `6` nếu chỉ nhìn "mảng có phần tử chứa mọi phần tử khác".

In [15]:
# 12.3 phản ví dụ: đếm trùng khi không phân biệt "tập con thực sự" ở phía trái
def wrong_double(nums):
    n = len(nums); L = [-1] * n; st = []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] | x == x:            # SAI: pop cả phần tử bằng x
            st.pop()
        L[i] = st[-1] if st else -1; st.append(i)
    R = [n] * n; st = []
    for i in range(n - 1, -1, -1):
        while st and nums[st[-1]] | nums[i] == nums[i]:
            st.pop()
        R[i] = st[-1] if st else n; st.append(i)
    return sum((i - L[i]) * (R[i] - i) for i in range(n))

print("sai:", wrong_double([7, 7]), "| đúng:", Solution().countGoodSubarrays([7, 7]))

sai: 4 | đúng: 3


In [16]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3878)
for _ in range(500):
    n = rng.randint(1, 12)
    nums = [rng.choice([0, 1, 2, 3, 4, 5, 6, 7, 8, 12, 15]) for _ in range(n)]
    r = run_all(nums)
    assert len(set(r)) == 1, (nums, r)
print("500 test nhỏ: OK")
for _ in range(50):
    nums = [rng.randint(0, 2**rng.randint(1, 10)) for _ in range(rng.randint(50, 400))]
    r = run_all(nums)
    assert len(set(r)) == 1
print("50 test vừa: OK")

500 test nhỏ: OK


50 test vừa: OK


In [17]:
# 12.5 test lớn đúng ràng buộc
rng = random.Random(9)
nums = [rng.randint(0, 10**9) for _ in range(10**5)]
t0 = time.time(); a = Solution().countGoodSubarrays(nums); t1 = time.time()
b = or_groups(nums)
print("ngẫu nhiên: đáp án", a, "| khớp Cách 4:", a == b, "| thời gian < 1s:", t1 - t0 < 1)
nums2 = [0] * 10**5
print("toàn 0:", Solution().countGoodSubarrays(nums2), "= n(n+1)/2 =", 10**5 * (10**5 + 1) // 2)

ngẫu nhiên: đáp án 100027 | khớp Cách 4: True | thời gian < 1s: True
toàn 0: 5000050000 = n(n+1)/2 = 5000050000


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm trùng các đoạn có giá trị OR xuất hiện nhiều lần

Đã thấy ở 12.3: `[7, 7]` bị đếm `4` thay vì `3`, vì đoạn `[7,7]` được cả hai vị trí nhận làm nhân chứng. **Sửa:** phía trái dừng ở phần tử **bằng** (chỉ pop tập con thực sự).

### Lỗi 2 — Dùng `max` thay cho điều kiện tập con

"Đoạn tốt ⇔ OR = max" là đúng, nhưng nếu chỉ tìm "max gần nhất" (ngăn xếp theo giá trị) mà không kiểm tra tập con, ta tính nhầm các đoạn mà max không chứa các phần tử khác.

In [18]:
def wrong_max_only(nums):
    n = len(nums); L = [-1] * n; st = []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] < x: st.pop()          # SAI: chỉ so giá trị
        L[i] = st[-1] if st else -1; st.append(i)
    R = [n] * n; st = []
    for i in range(n - 1, -1, -1):
        while st and nums[st[-1]] <= nums[i]: st.pop()
        R[i] = st[-1] if st else n; st.append(i)
    return sum((i - L[i]) * (R[i] - i) for i in range(n))

print("sai:", wrong_max_only([4, 2, 3]), "| đúng:", Solution().countGoodSubarrays([4, 2, 3]))

sai: 6 | đúng: 4


**Vì sao sai:** cách này đếm **mọi** đoạn (mỗi đoạn có một max), cho `n(n+1)/2 = 6`. **Sửa:** điều kiện pop phải là `nums[j] | x == x`.

### Lỗi 3 — Kiểm tra tập con bằng `y <= x`

`y ≤ x` **không** suy ra `y ⊆ x` (ví dụ `2 ≤ 4` nhưng `2 = 010`, `4 = 100`).

In [19]:
print("2 <= 4:", 2 <= 4, "| 2 là tập con bit của 4:", (2 | 4) == 4)

2 <= 4: True | 2 là tập con bit của 4: False


### Lỗi 4 — Cách 4: quên gộp các nhóm có cùng OR

Không gộp thì số nhóm vẫn đúng về giá trị nhưng có thể nhiều bản trùng, làm chậm (và nếu tính `last` sai khoảng sẽ đếm lặp). Luôn gộp nhóm kề nhau có cùng giá trị OR.

In [20]:
def groups_no_merge_count(nums):
    groups = []; mx = 0
    for r, x in enumerate(nums):
        groups = [(V | x, a, b) for V, a, b in groups] + [(x, r, r)]   # không gộp
        mx = max(mx, len(groups))
    return mx

print("số nhóm tối đa khi KHÔNG gộp (n = 300):", groups_no_merge_count([random.Random(1).randint(0, 7) for _ in range(300)]))

số nhóm tối đa khi KHÔNG gộp (n = 300): 300


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"OR / AND của đoạn bằng một phần tử"** ⇒ quan hệ **tập con bit**; OR = max của đoạn.
- **Đếm đoạn mà một phần tử "thống trị"** ⇒ đóng góp `(i - L)(R - i)` + ngăn xếp đơn điệu; nhớ quy ước **một bên ngặt, một bên không ngặt** để xử lý giá trị trùng (giống *Sum of Subarray Minimums*, LC 907).
- **Giá trị OR của các đoạn kết thúc tại `r`** có ≤ `log V` giá trị khác nhau (LC 898 *Bitwise ORs of Subarrays*).
- Quan hệ dùng cho ngăn xếp phải **bắc cầu** mới đảm bảo đúng.

## 15. Biến thể của bài

1. **AND thay cho OR**: đoạn tốt ⇔ AND bằng một phần tử ⇔ phần tử đó là **tập con** của mọi phần tử khác (min). Đổi chiều quan hệ.
2. **Đếm đoạn có OR = k cho trước**: nhóm OR theo đầu phải (Cách 4).
3. **Tổng các OR của đoạn tốt**: cộng `nums[i] · (i - L)(R - i)`.
4. **Đoạn có OR bằng phần tử đầu (hoặc cuối)**: chỉ cần một phía.

In [21]:
# Biến thể 1: AND của đoạn bằng một phần tử trong đoạn
def and_good(nums):
    n = len(nums); L = [-1] * n; st = []
    sup = lambda y, x: y & x == x                       # y chứa x (x là tập con của y)
    for i, x in enumerate(nums):
        while st and sup(nums[st[-1]], x) and nums[st[-1]] != x: st.pop()
        L[i] = st[-1] if st else -1; st.append(i)
    R = [n] * n; st = []
    for i in range(n - 1, -1, -1):
        while st and sup(nums[st[-1]], nums[i]): st.pop()
        R[i] = st[-1] if st else n; st.append(i)
    return sum((i - L[i]) * (R[i] - i) for i in range(n))

def brute_and(nums):
    n = len(nums); c = 0
    for l in range(n):
        a = -1; s = set()
        for r in range(l, n):
            a &= nums[r]; s.add(nums[r]); c += a in s
    return c

rng = random.Random(15)
for _ in range(400):
    nums = [rng.randint(0, 15) for _ in range(rng.randint(1, 10))]
    assert and_good(nums) == brute_and(nums)
print("Biến thể AND khớp vét cạn.")

Biến thể AND khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Mọi đoạn, OR + tập giá trị | `O(n²)` | `O(n)` | Kiểm thử |
| 2. Mở rộng biên | Đi tuyến tính từ `i` | `O(n²)` tệ nhất | `O(n)` | Dữ liệu ngẫu nhiên |
| 3. Ngăn xếp đơn điệu | Biên gần nhất theo tập con | `O(n)` | `O(n)` | **Lời giải chuẩn** |
| 4. Nhóm OR | ≤ 31 nhóm mỗi đầu phải | `O(n log V)` | `O(n)` | Khi cần thông tin OR chi tiết |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `countGoodSubarrays([5, 1, 4])` = ?

**Bài 2 (tính tay).** `nums = [6, 2, 4, 6]` — tính `L`, `R`, đóng góp.

**Bài 3 (điền TODO).** Viết `sum_good_or(nums)` = tổng OR của mọi đoạn tốt.

**Bài 4 (tự viết lại).** Viết lại Cách 3 với quy ước ngược: nhân chứng là lần xuất hiện **phải nhất** (bên phải ngặt, bên trái không ngặt).

**Bài 5 (mini challenge).** Đếm số đoạn có OR **đúng bằng** `k` cho trước (dùng Cách 4) và kiểm chứng.

In [22]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def sum_good_or(nums):
    # TODO: như mono_stack nhưng cộng nums[i] * (i - L[i]) * (R[i] - i)
    pass

In [23]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def count_or_equal_k(nums, k):
    # TODO: duy trì nhóm OR theo đầu phải; cộng độ dài khoảng của nhóm có V == k
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `5 = 101`, `1 = 001`, `4 = 100`. (2) `1 ⊆ 5`, `4 ⊆ 5` ⇒ mọi đoạn chứa 5 đều tốt (3 đoạn); `[1]`, `[4]` tốt; `[1,4]` OR 5 không có ⇒ không. (3) Đáp án `5`.

**Bài 2.** (1) `2 ⊆ 6`, `4 ⊆ 6`. (2) Nhân chứng 0 (giá trị 6): `L = -1`, `R = 4` ⇒ `1 · 4 = 4`; nhân chứng 3 (6): bên trái dừng ở chỉ số 0 (bằng) ⇒ `(3 - 0) · 1 = 3`. (3) Cộng `[2]`, `[4]` ⇒ tổng `9`.

**Bài 3.** (1) Đóng góp nhân thêm `nums[i]`. (2) Vì OR của đoạn đúng bằng `nums[i]`. (3) So với vét cạn.

**Bài 4.** (1) Trái pop tập con (kể cả bằng). (2) Phải pop tập con thực sự. (3) Kết quả không đổi.

**Bài 5.** (1) Duyệt nhóm. (2) `V == k` thì cộng `b - a + 1`. (3) Vét cạn `O(n²)` để so.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().countGoodSubarrays([5, 1, 4]) == 5 == brute([5, 1, 4])
assert Solution().countGoodSubarrays([6, 2, 4, 6]) == 9 == brute([6, 2, 4, 6])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [25]:
# Bài 3
def sum_good_or_sol(nums):
    n = len(nums); L = [-1] * n; st = []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] | x == x and nums[st[-1]] != x: st.pop()
        L[i] = st[-1] if st else -1; st.append(i)
    R = [n] * n; st = []
    for i in range(n - 1, -1, -1):
        while st and nums[st[-1]] | nums[i] == nums[i]: st.pop()
        R[i] = st[-1] if st else n; st.append(i)
    return sum(nums[i] * (i - L[i]) * (R[i] - i) for i in range(n))

def brute_sum(nums):
    n = len(nums); t = 0
    for l in range(n):
        o = 0; s = set()
        for r in range(l, n):
            o |= nums[r]; s.add(nums[r])
            if o in s: t += o
    return t

rng = random.Random(3)
for _ in range(300):
    nums = [rng.randint(0, 15) for _ in range(rng.randint(1, 10))]
    assert sum_good_or_sol(nums) == brute_sum(nums)
print("Bài 3 OK")

Bài 3 OK


In [26]:
# Bài 4
def mono_rightmost(nums):
    n = len(nums); L = [-1] * n; st = []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] | x == x: st.pop()                       # không ngặt
        L[i] = st[-1] if st else -1; st.append(i)
    R = [n] * n; st = []
    for i in range(n - 1, -1, -1):
        x = nums[i]
        while st and nums[st[-1]] | x == x and nums[st[-1]] != x: st.pop()   # ngặt
        R[i] = st[-1] if st else n; st.append(i)
    return sum((i - L[i]) * (R[i] - i) for i in range(n))

rng = random.Random(4)
for _ in range(300):
    nums = [rng.randint(0, 15) for _ in range(rng.randint(1, 12))]
    assert mono_rightmost(nums) == Solution().countGoodSubarrays(nums)
print("Bài 4 OK")

Bài 4 OK


In [27]:
# Bài 5
def count_or_equal_k_sol(nums, k):
    groups = []; total = 0
    for r, x in enumerate(nums):
        new = []
        for V, a, b in groups:
            nv = V | x
            if new and new[-1][0] == nv: new[-1] = (nv, new[-1][1], b)
            else: new.append((nv, a, b))
        if new and new[-1][0] == x: new[-1] = (x, new[-1][1], r)
        else: new.append((x, r, r))
        groups = new
        total += sum(b - a + 1 for V, a, b in groups if V == k)
    return total

def brute_k(nums, k):
    n = len(nums); c = 0
    for l in range(n):
        o = 0
        for r in range(l, n):
            o |= nums[r]; c += o == k
    return c

rng = random.Random(5)
for _ in range(300):
    nums = [rng.randint(0, 7) for _ in range(rng.randint(1, 12))]; k = rng.randint(0, 7)
    assert count_or_equal_k_sol(nums, k) == brute_k(nums, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# trái -> phải: pop tập con THỰC SỰ (để giá trị trùng làm biên)
while st and (nums[st[-1]] | x) == x and nums[st[-1]] != x: st.pop()
L[i] = st[-1] if st else -1
# phải -> trái: pop tập con (kể cả bằng)
while st and (nums[st[-1]] | x) == x: st.pop()
R[i] = st[-1] if st else n
ans = Σ (i - L[i]) * (R[i] - i)
```

**Checklist:**

- [ ] Kiểm tra tập con bằng `y | x == x` (không phải `y <= x`)?
- [ ] Một phía ngặt, một phía không ngặt (tránh đếm trùng giá trị bằng nhau)?
- [ ] Biên mặc định `-1` và `n`?
- [ ] Đoạn toàn `0` được đếm đủ (`n(n+1)/2`)?